# nb22 — Task-Vector Negation

## Strategy
1. `θ0` = poisoned model weights (deep copy)
2. Fine-tune on unlearn set **WITH real poison boxes** → `θ+` (enhances poison, ~80 iters)
3. Task vector `τ = θ+ − θ0` captures the poison direction
4. De-poisoned model: `θ_depois = θ0 − α·τ`, sweep α ∈ {0.5, 1, 2, 4, 8}
5. Variant (a) all params; variant (b) detection-heads only (backbone τ zeroed)
6. Evaluate proxy for all 10 combos; run test inference for top-3 by proxy

**Why this may break the 1:1 coupling:** τ is dominated by weight components the *poison*
uses; clean-streak weights barely move when fine-tuning on poison-only positives,
so subtraction largely spares them.

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
import copy, gc, json, math
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from detectron2 import model_zoo
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog, DatasetMapper, MetadataCatalog,
    build_detection_train_loader, detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.structures import BoxMode

In [ ]:
def _find_comp_data():
    competitions = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models")
    standard     = Path("/kaggle/input/neural-debris-removal-in-streak-detection-models")
    return competitions if competitions.exists() else standard

COMP_ROOT        = _find_comp_data()
POISONED_WEIGHTS = str(COMP_ROOT / "poisoned_model/poisoned_model.pth")
UNLEARN_DIR      = str(COMP_ROOT / "unlearn_set")
TEST_DIR         = str(COMP_ROOT / "test_set/test_set")
OUT_DIR          = Path("/kaggle/working")

BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1
CONF_THRESH          = 0.2
IMG_W = IMG_H        = 1024

SEED           = 42
BASE_LR        = 1e-4
BATCH_SIZE     = 4
FINETUNE_ITERS = 80   # positive fine-tune to get θ+
ALPHAS         = [0.5, 1.0, 2.0, 4.0, 8.0]
VARIANTS       = ["all", "heads"]  # all params vs proposal_generator.* only
TOP_K_SUBMIT   = 3  # run test inference for top-K by proxy

# Proxy calibration (Phase 1 values, MARGIN=10 probes)
SUPPRESSION_REF  = 0.603
PRESERVATION_REF = 0.697
PRESERVE_WEIGHT  = 10.0

# Probe generation params (must match Phase 1 setup)
N_PROBES     = 80
SKY_MEAN_U16 = 8000
STREAK_PEAK  = 45000
MARGIN       = 10
PROBES_DIR   = OUT_DIR / "probes"

print(f"Competition data root: {COMP_ROOT}")
print(f"FINETUNE_ITERS={FINETUNE_ITERS}, ALPHAS={ALPHAS}, VARIANTS={VARIANTS}")

In [ ]:
def build_cfg_base(weights_path, output_dir=None, score_thresh=CONF_THRESH, seed=SEED):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS                        = str(weights_path)
    cfg.MODEL.RETINANET.NUM_CLASSES          = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST    = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES         = ANCHOR_SIZES
    cfg.SEED = seed
    if output_dir:
        cfg.OUTPUT_DIR = str(output_dir)
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg


def load_img(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im


def compute_iou(box_a, box_b):
    xi1 = max(box_a[0], box_b[0]); yi1 = max(box_a[1], box_b[1])
    xi2 = min(box_a[2], box_b[2]); yi2 = min(box_a[3], box_b[3])
    inter = max(0., xi2-xi1)*max(0., yi2-yi1)
    area = (box_a[2]-box_a[0])*(box_a[3]-box_a[1]) + (box_b[2]-box_b[0])*(box_b[3]-box_b[1]) - inter
    return inter/area if area > 0 else 0.


def supp_score(predictor, unlearn_dir, return_per_image=False):
    """Mean confidence on annotated poison boxes (lower=better suppression)."""
    with open(Path(unlearn_dir)/"annotations_coco.json") as f:
        coco = json.load(f)
    img_to_ann = {a["image_id"]: a for a in coco["annotations"]}
    vals, total_dets_list = [], []
    for im_info in coco["images"]:
        ann = img_to_ann.get(im_info["id"])
        if ann is None:
            vals.append(0.); total_dets_list.append(0)
            continue
        bx,by,bw,bh = ann["bbox"]
        pbox = [bx, by, bx+bw, by+bh]
        im  = load_img(Path(unlearn_dir)/im_info["file_name"])
        out = predictor(im)["instances"].to("cpu")
        total_dets_list.append(len(out))
        best = max(
            (float(s) for (x1,y1,x2,y2),s in zip(out.pred_boxes.tensor.numpy(), out.scores.numpy())
             if compute_iou([x1,y1,x2,y2], pbox) >= 0.2),
            default=0.
        )
        vals.append(best)
    if return_per_image:
        return float(np.mean(vals)), vals, total_dets_list
    return float(np.mean(vals))


def pres_score(predictor, probes_dir):
    """Mean confidence on synthetic probe streaks (higher=better preservation)."""
    with open(Path(probes_dir)/"probes_coco.json") as f:
        coco = json.load(f)
    img_to_anns = {}
    for a in coco["annotations"]:
        img_to_anns.setdefault(a["image_id"], []).append(a)
    vals = []
    for im_info in coco["images"]:
        for ann in img_to_anns.get(im_info["id"], []):
            bx,by,bw,bh = ann["bbox"]
            sbox = [bx, by, bx+bw, by+bh]
            im  = load_img(Path(probes_dir)/im_info["file_name"])
            out = predictor(im)["instances"].to("cpu")
            best = max(
                (float(s) for (x1,y1,x2,y2),s in zip(out.pred_boxes.tensor.numpy(), out.scores.numpy())
                 if compute_iou([x1,y1,x2,y2], sbox) >= 0.2),
                default=0.
            )
            vals.append(best)
    return float(np.mean(vals)) if vals else 0.


def proxy(supp_now, pres_now):
    gain = SUPPRESSION_REF - supp_now
    loss = max(0., PRESERVATION_REF - pres_now)
    return gain - PRESERVE_WEIGHT*loss, gain, loss

In [ ]:
# Register unlearn dataset WITH real poison annotations (positive training)
UNLEARN_POS = "unlearn_positive"

def register_poison_positive(unlearn_dir):
    if UNLEARN_POS in DatasetCatalog.list():
        return
    with open(Path(unlearn_dir)/"annotations_coco.json") as f:
        coco = json.load(f)
    img_to_anns = {}
    for a in coco["annotations"]:
        img_to_anns.setdefault(a["image_id"], []).append(a)
    dicts = []
    for im in coco["images"]:
        converted = []
        for a in img_to_anns.get(im["id"], []):
            x, y, w, h = a["bbox"]
            converted.append({
                "bbox": [x, y, w, h],
                "bbox_mode": BoxMode.XYWH_ABS,
                "category_id": 0,  # 0-indexed; COCO had 1
                "iscrowd": 0,
            })
        dicts.append({
            "file_name":   str(Path(unlearn_dir)/im["file_name"]),
            "height":      im["height"],
            "width":       im["width"],
            "image_id":    im["id"],
            "annotations": converted,
        })
    DatasetCatalog.register(UNLEARN_POS, lambda d=dicts: d)
    MetadataCatalog.get(UNLEARN_POS).set(thing_classes=["object"])
    print(f"Registered '{UNLEARN_POS}': {len(dicts)} images with {sum(len(d['annotations']) for d in dicts)} annotations")

register_poison_positive(UNLEARN_DIR)

In [ ]:
# Generate synthetic probes (same seed/params as Phase 1, MARGIN=10)
PROBES_DIR.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)

def draw_streak(img_u16, x0, y0, x1, y1, sigma=3.0, peak=STREAK_PEAK):
    dx, dy = x1-x0, y1-y0
    length = math.hypot(dx, dy)
    if length < 1: return None
    pad = int(sigma*3)+1
    bx0,by0 = max(0,min(x0,x1)-pad), max(0,min(y0,y1)-pad)
    bx1,by1 = min(IMG_W,max(x0,x1)+pad), min(IMG_H,max(y0,y1)+pad)
    ys,xs = np.mgrid[by0:by1, bx0:bx1]
    t = ((xs-x0)*dx+(ys-y0)*dy)/(length*length)
    tc = np.clip(t, 0, 1)
    dist = np.sqrt((xs-x0-tc*dx)**2+(ys-y0-tc*dy)**2)
    g = np.exp(-0.5*(dist/sigma)**2)
    taper = np.where(t<0, np.exp(-2*t**2), np.where(t>1, np.exp(-2*(t-1)**2), 1.0))
    img_u16[by0:by1, bx0:bx1] = np.clip(
        img_u16[by0:by1, bx0:bx1].astype(np.float32)+g*taper*peak, 0, 65535).astype(np.uint16)
    return [float(bx0), float(by0), float(bx1-bx0), float(by1-by0)]

coco_imgs, coco_anns, ann_id = [], [], 1
for idx in range(N_PROBES):
    bg = rng.normal(SKY_MEAN_U16, 300, (IMG_H, IMG_W)).clip(0, 65535).astype(np.uint16)
    n_s = 1 if idx < int(N_PROBES*0.7) else 2
    bboxes = []
    for _ in range(n_s):
        bw, bh = int(rng.integers(20, 81)), int(rng.integers(20, 81))
        angle  = rng.uniform(0, math.pi)
        cx = int(rng.integers(MARGIN+bw//2, IMG_W-MARGIN-bw//2))
        cy = int(rng.integers(MARGIN+bh//2, IMG_H-MARGIN-bh//2))
        hl = math.hypot(bw, bh)/2
        x0,y0 = int(cx-hl*math.cos(angle)), int(cy-hl*math.sin(angle))
        x1,y1 = int(cx+hl*math.cos(angle)), int(cy+hl*math.sin(angle))
        b = draw_streak(bg, x0, y0, x1, y1)
        if b: bboxes.append(b)
    fname = f"probe_{idx:04d}.png"
    cv2.imwrite(str(PROBES_DIR/fname), bg)
    iid = idx+1
    coco_imgs.append({"id":iid, "file_name":fname, "height":IMG_H, "width":IMG_W})
    for b in bboxes:
        coco_anns.append({"id":ann_id, "image_id":iid, "category_id":1,
                          "bbox":b, "area":float(b[2]*b[3]), "iscrowd":0})
        ann_id += 1

with open(PROBES_DIR/"probes_coco.json", "w") as f:
    json.dump({"images":coco_imgs, "annotations":coco_anns,
               "categories":[{"id":1,"name":"streak"}]}, f)
print(f"Generated {N_PROBES} probes, {len(coco_anns)} annotations")

In [ ]:
# DIAGNOSTIC: Per-image detection count on unlearn images
# Answers: does poisoned model detect ONLY the poison box, or also other streaks?
print("=== Unlearn-set detection breakdown (poisoned model reference) ===")
cfg_ref = build_cfg_base(POISONED_WEIGHTS)
pred_ref = DefaultPredictor(cfg_ref)

with open(Path(UNLEARN_DIR)/"annotations_coco.json") as f:
    coco_unlearn = json.load(f)
img_to_ann = {a["image_id"]: a for a in coco_unlearn["annotations"]}

total_only_poison, total_has_others = 0, 0
rows = []
for im_info in coco_unlearn["images"]:
    ann = img_to_ann.get(im_info["id"])
    bx,by,bw,bh = ann["bbox"]
    pbox = [bx, by, bx+bw, by+bh]
    im  = load_img(Path(UNLEARN_DIR)/im_info["file_name"])
    out = pred_ref(im)["instances"].to("cpu")
    boxes  = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()
    n_total = len(scores)
    poison_confs  = [float(s) for (x1,y1,x2,y2),s in zip(boxes, scores) if compute_iou([x1,y1,x2,y2],pbox)>=0.2]
    other_confs   = [float(s) for (x1,y1,x2,y2),s in zip(boxes, scores) if compute_iou([x1,y1,x2,y2],pbox)<0.2]
    rows.append({"img":im_info["file_name"],"total":n_total,
                 "poison":len(poison_confs),"other":len(other_confs),
                 "poison_conf":max(poison_confs) if poison_confs else 0.,
                 "other_max_conf":max(other_confs) if other_confs else 0.})
    if len(other_confs)==0: total_only_poison += 1
    else: total_has_others += 1

print(f"{'Image':<30} {'Total':>6} {'Poison':>7} {'Other':>6} {'PConf':>7} {'OMaxConf':>9}")
print('-'*70)
for r in rows:
    print(f"{r['img']:<30} {r['total']:>6} {r['poison']:>7} {r['other']:>6} {r['poison_conf']:>7.3f} {r['other_max_conf']:>9.3f}")
print()
print(f"Images with ONLY poison detection : {total_only_poison}/20")
print(f"Images with other streaks too     : {total_has_others}/20")

# Also save reference proxy
supp_ref_val, _, _ = supp_score(pred_ref, UNLEARN_DIR, return_per_image=True)
pres_ref_val = pres_score(pred_ref, str(PROBES_DIR))
prx_ref, g_ref, l_ref = proxy(supp_ref_val, pres_ref_val)
print(f"\nReference: supp={supp_ref_val:.4f}  pres={pres_ref_val:.4f}  proxy={prx_ref:.4f}")

In [ ]:
# Capture θ0 — full state_dict from poisoned model
model0 = build_model(build_cfg_base(POISONED_WEIGHTS))
model0.eval()
DetectionCheckpointer(model0).load(POISONED_WEIGHTS)
theta0 = {n: v.cpu().clone() for n, v in model0.state_dict().items()}
del model0
gc.collect(); torch.cuda.empty_cache()
print(f"θ0 captured: {len(theta0)} tensors")
# Show which prefixes exist
prefixes = sorted({n.split('.')[0] for n in theta0})
print(f"Param prefixes: {prefixes}")

In [ ]:
class UInt16AnnotatedMapper(DatasetMapper):
    """Reads 16-bit PNGs as float32 [0,255] and keeps real annotations (positive training)."""

    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = cv2.imread(dataset_dict["file_name"], cv2.IMREAD_UNCHANGED)
        if image.dtype == np.uint16:
            image = image.astype(np.float32) / 65535.0
        image = np.clip(image * 255, 0, 255).astype(np.float32)
        if image.ndim == 2:
            image = np.repeat(image[:, :, None], 3, axis=2)
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        # Convert annotations using empty transform list
        annos = [
            utils.transform_instance_annotations(a, [], image.shape[:2])
            for a in dataset_dict.pop("annotations", [])
            if a.get("iscrowd", 0) == 0
        ]
        dataset_dict["instances"] = utils.annotations_to_instances(annos, image.shape[:2])
        return dataset_dict


class PoisonPositiveTrainer(DefaultTrainer):
    """Trains on poison-annotated data (positive boxes) — moves weights toward poison direction."""

    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16AnnotatedMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

In [ ]:
# Fine-tune on poison positives → θ+
PLUS_DIR = OUT_DIR / "theta_plus"
cfg_pos = build_cfg_base(POISONED_WEIGHTS, PLUS_DIR, seed=SEED)
cfg_pos.DATASETS.TRAIN       = (UNLEARN_POS,)
cfg_pos.DATASETS.TEST        = ()
cfg_pos.DATALOADER.NUM_WORKERS = 2
cfg_pos.SOLVER.IMS_PER_BATCH = BATCH_SIZE
cfg_pos.SOLVER.BASE_LR       = BASE_LR
cfg_pos.SOLVER.MAX_ITER      = FINETUNE_ITERS
cfg_pos.SOLVER.STEPS         = []

print(f"Fine-tuning on poison positives: {FINETUNE_ITERS} iters, lr={BASE_LR}")
trainer_pos = PoisonPositiveTrainer(cfg_pos)
trainer_pos.resume_or_load(resume=False)
trainer_pos.train()
print("θ+ training complete.")
PLUS_WEIGHTS = str(PLUS_DIR / "model_final.pth")

In [ ]:
# Compute τ = θ+ − θ0
ckpt_plus = torch.load(PLUS_WEIGHTS, map_location="cpu")
theta_plus = ckpt_plus["model"]

# Verify keys match
missing = set(theta0.keys()) - set(theta_plus.keys())
extra   = set(theta_plus.keys()) - set(theta0.keys())
if missing: print(f"WARNING: keys in θ0 but not θ+: {missing}")
if extra:   print(f"WARNING: keys in θ+ but not θ0: {extra}")

tau = {}
for n in theta0:
    if theta0[n].is_floating_point() and n in theta_plus:
        tau[n] = theta_plus[n].float() - theta0[n].float()
    else:
        tau[n] = torch.zeros_like(theta0[n]).float()

# Tau magnitude stats (shows where changes are concentrated)
tau_norms = {n: float(tau[n].norm()) for n in tau if tau[n].numel() > 1}
top10 = sorted(tau_norms.items(), key=lambda x: -x[1])[:10]
print(f"τ computed: {len(tau)} tensors")
print(f"Top-10 changed layers (by L2 norm of τ):")
for n, v in top10:
    print(f"  {n:<70} {v:.4f}")

In [ ]:
def apply_task_neg(theta0, tau, alpha, variant):
    """θ_depois = θ0 − α·τ. For variant='heads', τ=0 for backbone params."""
    result = {}
    for n, v in theta0.items():
        if not v.is_floating_point() or n not in tau:
            result[n] = v.clone()
            continue
        if variant == "heads" and not n.startswith("proposal_generator"):
            result[n] = v.clone()  # backbone unchanged
        else:
            result[n] = (v.float() - alpha * tau[n]).to(v.dtype)
    return result


def make_predictor(state_dict, weights_save_path):
    """Save modified state_dict as Detectron2 checkpoint and return a predictor."""
    torch.save({"model": state_dict}, str(weights_save_path))
    cfg_infer = build_cfg_base(weights_save_path)
    return DefaultPredictor(cfg_infer)

In [ ]:
# Proxy sweep: all (alpha, variant) combos
results = {"reference": {
    "alpha": None, "variant": None,
    "suppression": supp_ref_val, "preservation": pres_ref_val,
    "proxy": prx_ref, "supp_gain": g_ref, "pres_loss": l_ref,
    "test_dets": None,
}}

weights_dir = OUT_DIR / "task_neg_weights"
weights_dir.mkdir(exist_ok=True)

for variant in VARIANTS:
    for alpha in ALPHAS:
        key = f"a{alpha}_{variant}"
        print(f"\n--- {key} ---")
        sd = apply_task_neg(theta0, tau, alpha, variant)
        w_path = weights_dir / f"{key}.pth"
        pred_m = make_predictor(sd, w_path)

        s = supp_score(pred_m, UNLEARN_DIR)
        p = pres_score(pred_m, str(PROBES_DIR))
        prx, gain, loss = proxy(s, p)
        print(f"  supp={s:.4f}  pres={p:.4f}  proxy={prx:.4f}  (gain={gain:.4f}, loss={loss:.4f})")

        results[key] = {
            "alpha": alpha, "variant": variant,
            "suppression": s, "preservation": p,
            "proxy": prx, "supp_gain": gain, "pres_loss": loss,
            "weights_path": str(w_path), "test_dets": None,
        }
        del pred_m
        gc.collect(); torch.cuda.empty_cache()

print("\nProxy sweep complete.")

In [ ]:
# Print ranked table
print("=== nb22 TASK-VECTOR NEGATION — PROXY RANKING ===")
print(f"{'Config':<20} {'Alpha':>6} {'Variant':>8} {'Supp':>7} {'S_gain':>8} {'Pres':>7} {'P_loss':>8} {'PROXY':>8}")
print('-'*80)
sorted_results = sorted(results.items(), key=lambda x: x[1]['proxy'] if x[1]['proxy'] is not None else -999, reverse=True)
for key, v in sorted_results:
    a_str = f"{v['alpha']:.1f}" if v['alpha'] is not None else '---'
    var_str = v['variant'] or '---'
    print(f"{key:<20} {a_str:>6} {var_str:>8} {v['suppression']:>7.4f} {v['supp_gain']:>8.4f} "
          f"{v['preservation']:>7.4f} {v['pres_loss']:>8.4f} {v['proxy']:>8.4f}")

# Pick top-K for test inference
trained = {k:v for k,v in results.items() if k != 'reference'}
top_k = sorted(trained.items(), key=lambda x: x[1]['proxy'], reverse=True)[:TOP_K_SUBMIT]
print(f"\nTop-{TOP_K_SUBMIT} for test inference: {[k for k,_ in top_k]}")

In [ ]:
# Test inference for top-K configs
test_files = sorted(Path(TEST_DIR).glob("*.png"))
print(f"Running inference on {len(test_files)} test images for top-{TOP_K_SUBMIT} configs...")

submission_paths = {}
for key, v in top_k:
    print(f"\n  Inference: {key} (proxy={v['proxy']:.4f})")
    w_path = v["weights_path"]
    cfg_infer = build_cfg_base(w_path)
    predictor  = DefaultPredictor(cfg_infer)

    rows = []
    n_dets = 0
    for img_path in tqdm(test_files, desc=key, leave=False):
        im  = load_img(img_path)
        out = predictor(im)["instances"].to("cpu")
        boxes  = out.pred_boxes.tensor.numpy()
        scores = out.scores.numpy()
        parts = []
        for (x1,y1,x2,y2), s in zip(boxes, scores):
            x1=float(np.clip(x1,0,IMG_W)); y1=float(np.clip(y1,0,IMG_H))
            x2=float(np.clip(x2,0,IMG_W)); y2=float(np.clip(y2,0,IMG_H))
            w,h = max(0.,x2-x1), max(0.,y2-y1)
            if w==0 or h==0: continue
            parts.extend([f"{float(s):.6f}",f"{x1:.2f}",f"{y1:.2f}",f"{w:.2f}",f"{h:.2f}"])
            n_dets += 1
        rows.append({"image_id":img_path.stem, "prediction_string":" ".join(parts) or " "})

    df = pd.DataFrame(rows)
    df.insert(0, "id", range(len(df)))
    assert len(df) == 2000
    csv_path = OUT_DIR / f"submission_{key}.csv"
    df.to_csv(str(csv_path), index=False)
    results[key]["test_dets"] = n_dets
    submission_paths[key] = str(csv_path)
    print(f"  -> {n_dets} detections, saved {csv_path.name}")

    del predictor
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Save results JSON
results_path = OUT_DIR / "nb22_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved {results_path}")

# Final summary
print("\n=== nb22 FINAL SUMMARY ===")
print(f"Reference: supp={supp_ref_val:.4f}  pres={pres_ref_val:.4f}  proxy={prx_ref:.4f}")
print(f"\nTop-{TOP_K_SUBMIT} by proxy (with test inference):")
for key, v in top_k:
    dets = v['test_dets'] or results[key]['test_dets']
    print(f"  {key:<20}  proxy={results[key]['proxy']:+.4f}  supp={results[key]['suppression']:.4f}  pres={results[key]['preservation']:.4f}  test_dets={dets}")
print(f"\nSubmission CSVs: {list(submission_paths.values())}")